# Session 3 — Instructor Solutions
**Engineering Data Analysis with pandas | Dr. Nuha Aljuneidi**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
data = {
    "run": range(1, 13),
    "flow_L_min": [2,2,2,4,4,4,6,6,6,8,8,8],
    "T_h_in_C": [60.2,60.0,60.1,60.3,60.1,60.2,60.4,60.2,60.3,60.5,60.4,60.3],
    "T_h_out_C": [42.8,43.1,42.9,45.4,45.1,45.3,47.2,47.0,47.1,49.0,None,48.8],
    "T_c_in_C": [20.1,20.0,20.2,20.1,20.0,20.1,20.2,20.1,20.0,20.2,20.1,20.2],
    "T_c_out_C": [36.9,36.7,36.8,34.6,34.8,70.0,32.9,33.1,33.0,31.8,31.9,31.7],
    "pressure_drop_kPa": [3.1,3.0,3.2,7.6,7.8,7.7,13.9,14.2,14.0,22.4,22.1,22.6]
}
df_raw = pd.DataFrame(data)
df_raw

In [ ]:
display(df_raw.head())
df_raw.info()
display(df_raw.describe())
display(df_raw.isna().sum().rename("missing"))

Run 11 has a missing hot-outlet temperature. Run 6 has an impossible cold-outlet temperature above the hot inlet.

In [ ]:
df = df_raw.copy()
required = ["T_h_in_C", "T_h_out_C", "T_c_in_C", "T_c_out_C"]
valid = (df[required].notna().all(axis=1)
         & df["T_c_out_C"].between(df["T_c_in_C"], df["T_h_in_C"], inclusive="neither")
         & df["T_h_out_C"].between(df["T_c_in_C"], df["T_h_in_C"], inclusive="neither"))
df = df.loc[valid].copy()
df

In [ ]:
cp = 4180.0
df["m_dot_kg_s"] = df["flow_L_min"] / 60.0
df["Q_hot_kW"] = df["m_dot_kg_s"] * cp * (df["T_h_in_C"] - df["T_h_out_C"]) / 1000
df["Q_cold_kW"] = df["m_dot_kg_s"] * cp * (df["T_c_out_C"] - df["T_c_in_C"]) / 1000
df["Q_avg_kW"] = (df["Q_hot_kW"] + df["Q_cold_kW"]) / 2
df["imbalance_pct"] = abs(df["Q_hot_kW"] - df["Q_cold_kW"]) / df["Q_avg_kW"] * 100
df["C_min_W_K"] = df["m_dot_kg_s"] * cp
df["effectiveness"] = df["Q_avg_kW"] * 1000 / (df["C_min_W_K"] * (df["T_h_in_C"] - df["T_c_in_C"]))
display(df.round(3))

In [ ]:
df_reliable = df.loc[df["imbalance_pct"] <= 10].copy()
print(f"Raw: {len(df_raw)} | Temperature-valid: {len(df)} | Reliable: {len(df_reliable)}")

In [ ]:
summary = (df_reliable.groupby("flow_L_min")
 .agg(n=("run", "count"),
      effectiveness_mean=("effectiveness", "mean"),
      effectiveness_std=("effectiveness", "std"),
      Q_avg_mean_kW=("Q_avg_kW", "mean"),
      pressure_drop_mean_kPa=("pressure_drop_kPa", "mean"))
 .reset_index())
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.5), constrained_layout=True)
ax.errorbar(summary["flow_L_min"], summary["effectiveness_mean"],
            yerr=summary["effectiveness_std"], marker="o", capsize=5,
            linewidth=2, color="#176B87")
ax.set_xlabel("Volume flow rate (L/min)")
ax.set_ylabel("Heat-exchanger effectiveness (–)")
ax.set_title("Heat-Exchanger Performance from Repeated Tests")
plt.show()

## Interpretation

Effectiveness decreases as flow rate rises in this dataset, while total heat-transfer rate and pressure drop increase. A reasonable recommendation is 4–6 L/min when increased heat duty justifies the added pumping penalty; 2 L/min is preferable when effectiveness and low pressure loss dominate. Students should cite their grouped values and explicitly state the design priority.